In [1]:
import requests

from src.server.models.message import Message

base_url = "http://localhost:8080"

# Create tasksmanager
tasksmanager_data = {
    "name": "Astronomy Expert",
    "model": "gpt-4",
    "description": "TasksManager for astronomy research",
}

tasksmanager_response = requests.post(f"{base_url}/tasksmanager", json=tasksmanager_data)
assert tasksmanager_response.status_code == 201
tasksmanager_id = tasksmanager_response.json()["id"]

# Create task
system_message = Message(
    role="system",
    content="You are an astronomy expert ready to answer questions about the solar system."
)

request_data = {
    "company_description": "A freelance comedian",
    "company_domain": "joking.com",
    "hiring_needs": "We are looking for a joke agent to join our team.",
    "specific_benefits": "Remote work, flexible hours, stock options",
    "agents": [
        {
            "role": "Comedian",
            "goal": "Tell a joke",
            "backstory": "48 year old male doing stand-up comedy for over 15 years started in new york and now lives in LA",
            "tools": [
                {"name": "WebsiteSearchTool", "description": "Tool for searching websites"},
                {"name": "SeperDevTool", "description": "Development tool for data separation"},
            ],
        }
    ],
    "tasks": [
        {
            "description": "make a joke about monkeys",
            "expected_output": "the joke should be child friendly and not insult anyone",
            "agent_role": "Comedian"
        },
    ]
}

response = requests.post(f"{base_url}/agent/crewai", json=request_data)
assert response.status_code == 200
data = response.json()
print(data)

{'message': 'Crew created', 'crewai_config': {'id': 'ce2ef003-5b52-415b-936b-9bd926f848be', 'create_crew': {'company_description': 'A freelance comedian', 'company_domain': 'joking.com', 'hiring_needs': 'We are looking for a joke agent to join our team.', 'specific_benefits': 'Remote work, flexible hours, stock options', 'agents': [{'role': 'Comedian', 'goal': 'Tell a joke', 'backstory': '48 year old male doing stand-up comedy for over 15 years started in new york and now lives in LA', 'tools': [{'name': 'WebsiteSearchTool', 'description': 'Tool for searching websites'}, {'name': 'SeperDevTool', 'description': 'Development tool for data separation'}]}], 'tasks': [{'description': 'make a joke about monkeys', 'expected_output': 'the joke should be child friendly and not insult anyone', 'agent_role': 'Comedian'}]}, 'created_at': '2024-07-16T14:53:18.388016', 'updated_at': None}}


In [4]:

task_data = {
    "name": "Solar System Questions",
    "description": "Answer questions about the solar system, alway anser distance in Miles.",
    "tasksmanager_id": tasksmanager_id,
    "messages": [system_message.dict()],
    "crewai_info_id": data['crewai_config']['id'],
}

task_response = requests.post(f"{base_url}/task", json=task_data)
assert task_response.status_code == 201
task_id = task_response.json()["id"]

# Run task with user message
user_message = Message(
    role="user",
    content="What is the distance from Earth to the Moon in kilometers?"
)
run_task = {
    'caller_id': "UnitTestCaller",
    'messages': [user_message.dict()]
}

run_task_response = requests.post(f"{base_url}/task/{tasksmanager_id}/{task_id}/run", json=run_task)